In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Composite DNA Decoder: Training & Evaluation - Eta-Based Variable Ratio
# ## Extended alphabet with variable mixture ratios (η=0.2, ℓ∈{-2,-1,0,1,2})

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID


# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


# =============================================================================
# CELL 3: CONFIGURATION & HYPERPARAMETERS
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
ERROR_MODEL = "erlich"

# ------------------- ETA-BASED ALPHABET PARAMETERS -------------------
ETA = 0.2
ELL_VALUES = [-2, -1, 0, 1, 2]

# Dataset parameters
NUM_SAMPLES = 100000
MAX_COVERAGE = 50

# Calculate vocab size
NUM_PURE_BASES = 4
NUM_TWO_MIX_PAIRS = 6
VOCAB_SIZE = NUM_PURE_BASES + NUM_TWO_MIX_PAIRS * len(ELL_VALUES)  # 34

# Error model specifications
ERROR_MODEL_SPECS = {
    "erlich": {"seq_length": 136, "name": "EZ17"},
    "grass": {"seq_length": 104, "name": "G15"},
    "organick": {"seq_length": 77, "name": "O17"}
}

# Build configuration
CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_name": ERROR_MODEL_SPECS[ERROR_MODEL]["name"],
    
    # Eta Parameters
    "eta": ETA,
    "ell_values": ELL_VALUES,
    "alphabet_mode": f"eta{ETA}",
    
    # Data Paths
    "dataset_dir": "./dataset",
    "dataset_name": f"dna_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_eta{ETA}",
    
    # Results directory
    "results_dir": f"./results_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_eta{ETA}",
    
    # Vocabulary
    "vocab_size": VOCAB_SIZE,
    
    # Sequence Parameters
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    
    # Experiment Parameters
    "coverage_levels": [1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 40, 50],
    
    # Model Architecture
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Training Parameters
    "batch_size": 500,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "patience": 10,
    "warmup_epochs": 10,
    "min_lr": 1e-6,
    
    # Reproducibility
    "seed": 42
}

# Complete dataset path
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{CONFIG['dataset_name']}_"
                          f"{NUM_SAMPLES}_{MAX_COVERAGE}.pkl")

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print(f"{'='*60}")
print(f"📋 CONFIGURATION")
print(f"{'='*60}")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Eta: {CONFIG['eta']}, Ell Values: {CONFIG['ell_values']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Theoretical Capacity: {np.log2(CONFIG['vocab_size']):.4f} bits/position")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"   Results Dir: {CONFIG['results_dir']}")
print(f"{'='*60}")


# =============================================================================
# CELL 4: SEED & REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")


# =============================================================================
# CELL 5: BUILD ETA-BASED ALPHABET MAPPINGS
# =============================================================================

def build_eta_based_symbol_to_idx(eta, ell_values):
    """Build symbol-to-index mapping for eta-based alphabet."""
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    pair_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']
    current_idx = 4
    
    for pair_name in pair_names:
        for ell in ell_values:
            if ell >= 0:
                symbol_name = f"{pair_name}_ell{ell}"
            else:
                symbol_name = f"{pair_name}_ell_neg{abs(ell)}"
            symbol_to_idx[symbol_name] = current_idx
            current_idx += 1
    
    return symbol_to_idx


def build_eta_based_ideal_vectors(eta, ell_values):
    """
    Build ideal frequency vectors for eta-based alphabet.
    
    Returns:
        torch.Tensor of shape (vocab_size, 4)
    """
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
    ]
    
    # Two-mix pairs: (vec_idx1, vec_idx2)
    pair_indices = [
        (0, 1),  # B1: A|C
        (0, 2),  # B2: A|G
        (0, 3),  # B3: A|T
        (1, 2),  # B4: C|G
        (1, 3),  # B5: C|T
        (2, 3),  # B6: G|T
    ]
    
    for idx1, idx2 in pair_indices:
        for ell in ell_values:
            prob1 = 0.5 + ell * eta
            prob2 = 0.5 - ell * eta
            
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[idx1] = prob1
            vec[idx2] = prob2
            ideal_vectors.append(vec)
    
    return torch.tensor(ideal_vectors, dtype=torch.float32)


# Build mappings
SYMBOL_TO_IDX = build_eta_based_symbol_to_idx(CONFIG["eta"], CONFIG["ell_values"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_eta_based_ideal_vectors(CONFIG["eta"], CONFIG["ell_values"]).to(device)

print(f"\n📊 Symbol Mappings (η={CONFIG['eta']}):")
print(f"   Total symbols: {len(SYMBOL_TO_IDX)}")
print(f"\n   Sample mappings (first 10):")
for i, (sym, idx) in enumerate(sorted(SYMBOL_TO_IDX.items(), key=lambda x: x[1])[:10]):
    vec = IDEAL_VECTORS[idx].cpu().numpy()
    print(f"   {sym:<16} {idx:<4} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}, {vec[3]:.2f}]")
print(f"   ... ({len(SYMBOL_TO_IDX) - 10} more)")


# =============================================================================
# CELL 6: DATA PREPROCESSING (Same as previous)
# =============================================================================
# def preprocess_cluster_to_matrix(cluster_reads, target_length): ...
# Copy from previous code

def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """
    Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix.
    """
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
            
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
                
    if num_reads > 0:
        profile_matrix /= num_reads
        
    return profile_matrix


# =============================================================================
# CELL 7: PYTORCH DATASET CLASS (Same structure, updated for eta)
# =============================================================================

class CompositeDNADatasetEta(Dataset):
    """PyTorch Dataset for Eta-Based Composite DNA data."""
    
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.metadata = raw_data['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        
        if self.limit_coverage is not None:
            actual_limit = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual_limit]
            
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)


# =============================================================================
# CELL 8: NEURAL NETWORK MODEL (Same as previous)
# =============================================================================
# class CompositeDecoderLSTM(nn.Module): ...
# Copy from previous code

class CompositeDecoderLSTM(nn.Module):
    """Bidirectional LSTM Decoder for Composite DNA."""
    
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        # x: (Batch, 4, L) -> (Batch, L, 4)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        # Return: (Batch, vocab_size, L)
        return logits.permute(0, 2, 1)


# =============================================================================
# CELL 9: BASELINE DECODERS (Same logic, works with any ideal_vectors)
# =============================================================================
# def min_distance_decoder(obs, ideal_vectors): ...
# def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01): ...
# def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01): ...
# Copy from previous code

def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder (L2 norm)."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)


# =============================================================================
# CELL 10: EARLY STOPPING CLASS (Same as previous)
# =============================================================================
# class EarlyStopping: ...
# Copy from previous code

class EarlyStopping:
    """Early stopping with patience and best model saving."""
    
    def __init__(self, patience=5, path='checkpoint.pt', verbose=True):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.path = path
        self.verbose = verbose
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f"      EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
            
    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f"      ✓ Val loss improved ({self.best_val_loss:.4f} → {val_loss:.4f}). Saving...")
        torch.save(model.state_dict(), self.path)
        self.best_val_loss = val_loss


# =============================================================================
# CELL 11: TRAINING FUNCTION (Same as previous)
# =============================================================================
# def train_model(model, train_loader, val_loader, config, weights_path, device): ...
# Copy from previous code

def train_model(model, train_loader, val_loader, config, weights_path, device):
    """Train the model with warmup + cosine annealing scheduler."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=config['warmup_epochs'])
    cosine_scheduler = CosineAnnealingLR(
        optimizer, T_max=config['epochs'] - config['warmup_epochs'], eta_min=config['min_lr']
    )
    scheduler = SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[config['warmup_epochs']]
    )
    
    early_stopper = EarlyStopping(patience=config['patience'], path=weights_path, verbose=True)
    
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    print(f"\n   🏋️ Training Configuration:")
    print(f"      Epochs: {config['epochs']}, Patience: {config['patience']}")
    print(f"      Warmup: {config['warmup_epochs']} epochs")
    print(f"      LR: {config['learning_rate']} → {config['min_lr']}")
    
    for epoch in range(config['epochs']):
        start_time = time.time()
        
        # Training
        model.train()
        train_loss_accum = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_accum += loss.item()
        avg_train_loss = train_loss_accum / len(train_loader)
        
        # Validation
        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss_accum += criterion(outputs, labels).item()
        avg_val_loss = val_loss_accum / len(val_loader)
        
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['lr'].append(current_lr)
        
        elapsed = time.time() - start_time
        print(f"   Epoch {epoch+1:03d}/{config['epochs']} | "
              f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.2e} | Time: {elapsed:.1f}s")
        
        scheduler.step()
        early_stopper(avg_val_loss, model)
        
        if early_stopper.early_stop:
            print(f"\n   🛑 Early stopping triggered at epoch {epoch+1}")
            break
    
    return history


# =============================================================================
# CELL 12: EVALUATION FUNCTION (Same as previous)
# =============================================================================
# def evaluate_all_decoders(model, loader, ideal_vectors, device): ...
# Copy from previous code

def evaluate_all_decoders(model, loader, ideal_vectors, device):
    """Evaluate all 4 decoders on the given data loader."""
    model.eval()
    
    correct = {'lstm': 0, 'mindist': 0, 'kl': 0, 'ml': 0}
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)
            
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            total += labels.numel()
            correct['lstm'] += (pred_lstm == labels).sum().item()
            correct['mindist'] += (pred_mindist == labels).sum().item()
            correct['kl'] += (pred_kl == labels).sum().item()
            correct['ml'] += (pred_ml == labels).sum().item()
    
    accuracies = {k: 100 * v / total for k, v in correct.items()}
    return accuracies


# =============================================================================
# CELL 13: FULL EXPERIMENT FOR SINGLE COVERAGE
# =============================================================================

def run_experiment_for_coverage(coverage_M, config, symbol_to_idx, ideal_vectors, device):
    """Run complete experiment for a single coverage level."""
    
    print(f"\n{'='*70}")
    print(f"🔬 EXPERIMENT FOR COVERAGE M = {coverage_M}")
    print(f"   Eta: {config['eta']}, Vocab Size: {config['vocab_size']}")
    print(f"{'='*70}")
    
    set_seed(config['seed'])
    
    full_ds = CompositeDNADatasetEta(
        config['dataset_path'], config['seq_length'], symbol_to_idx, limit_coverage=coverage_M
    )
    
    train_size = int(0.8 * len(full_ds))
    val_size = len(full_ds) - train_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)
    
    print(f"   📊 Data: {train_size:,} train | {val_size:,} validation")
    
    model = CompositeDecoderLSTM(config).to(device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🧠 Model: {config['vocab_size']} classes, {num_params:,} parameters")
    
    model_prefix = f"{config['error_name']}_eta{config['eta']}"
    best_weights_path = os.path.join(config['results_dir'], f"best_model_{model_prefix}_M{coverage_M}.pth")
    final_weights_path = os.path.join(config['results_dir'], f"final_model_{model_prefix}_M{coverage_M}.pth")
    history_path = os.path.join(config['results_dir'], f"training_history_{model_prefix}_M{coverage_M}.json")
    
    history = train_model(model, train_loader, val_loader, config, best_weights_path, device)
    
    torch.save(model.state_dict(), final_weights_path)
    print(f"   💾 Final model saved: {final_weights_path}")
    
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=4)
    
    print(f"\n   📈 Evaluating all decoders...")
    model.load_state_dict(torch.load(best_weights_path, map_location=device))
    
    accuracies = evaluate_all_decoders(model, val_loader, ideal_vectors, device)
    
    print(f"\n   ✅ RESULTS M={coverage_M} (η={config['eta']}):")
    print(f"      Bi-LSTM:         {accuracies['lstm']:.2f}%")
    print(f"      Min. Distance:   {accuracies['mindist']:.2f}%")
    print(f"      KL Divergence:   {accuracies['kl']:.2f}%")
    print(f"      Max. Likelihood: {accuracies['ml']:.2f}%")
    
    return accuracies, history


# =============================================================================
# CELL 14: PLOTTING FUNCTIONS (Same as previous)
# =============================================================================
# def plot_training_history(history, coverage_M, save_path, config): ...
# def plot_comparison_results(results, save_path, config): ...
# Copy from previous code

def plot_training_history(history, coverage_M, save_path, config):
    """Plot training and validation loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Training & Validation Loss (M={coverage_M}, η={config["eta"]})', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['lr'], 'g-', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Learning Rate', fontsize=12)
    ax2.set_title(f'Learning Rate Schedule (M={coverage_M})', fontsize=14)
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   📈 Training plot saved: {save_path}")


def plot_comparison_results(results, save_path, config):
    """Plot comparison of all decoders across coverage levels."""
    plt.figure(figsize=(12, 7))
    
    plt.plot(results['coverage'], results['lstm'], 
             'o-', lw=2.5, ms=8, c='#2ecc71', label='Bi-LSTM (Ours)')
    plt.plot(results['coverage'], results['mindist'], 
             's--', lw=2.5, ms=8, c='#e74c3c', label='Min. Distance')
    plt.plot(results['coverage'], results['kl'], 
             '^-.', lw=2.5, ms=8, c='#3498db', label='KL Divergence')
    plt.plot(results['coverage'], results['ml'], 
             'd:', lw=2.5, ms=8, c='#9b59b6', label='Max. Likelihood')
    
    title = (f"Composite DNA Decoding: η={config['eta']} ({config['vocab_size']} classes)\n"
             f"Error Model: {config['error_name']}, Seq Length: {config['seq_length']}")
    
    plt.xlabel("Coverage Depth (M)", fontsize=12)
    plt.ylabel("Symbol Accuracy (%)", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)
    plt.xticks(results['coverage'])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Comparison plot saved: {save_path}")


# =============================================================================
# CELL 15: VERIFY DATASET EXISTS
# =============================================================================

print("\n" + "="*70)
print("📦 LOADING DATASET")
print("="*70)

if not os.path.exists(CONFIG['dataset_path']):
    raise FileNotFoundError(
        f"\n❌ Dataset not found: {CONFIG['dataset_path']}\n"
        f"   Please run dataset generator first with:\n"
        f"   ETA = {CONFIG['eta']}, ELL_VALUES = {CONFIG['ell_values']}"
    )

with open(CONFIG['dataset_path'], 'rb') as f:
    data = pickle.load(f)

print(f"✅ Dataset loaded: {CONFIG['dataset_path']}")
print(f"   Samples: {len(data['data']):,}")
print(f"   Vocab Size: {data['metadata']['vocab_size']}")
print(f"   Eta: {data['metadata']['eta']}")


# =============================================================================
# CELL 16: MAIN EXECUTION - RUN ALL EXPERIMENTS
# =============================================================================

print("\n" + "="*70)
print("🚀 RUNNING EXPERIMENTS FOR ALL COVERAGE LEVELS")
print("="*70)
print(f"   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']}")
print(f"   Coverage Levels: {CONFIG['coverage_levels']}")

results = {
    'coverage': CONFIG['coverage_levels'],
    'lstm': [],
    'mindist': [],
    'kl': [],
    'ml': [],
    'config': {
        'error_model': CONFIG['error_model'],
        'error_name': CONFIG['error_name'],
        'seq_length': CONFIG['seq_length'],
        'eta': CONFIG['eta'],
        'ell_values': CONFIG['ell_values'],
        'vocab_size': CONFIG['vocab_size'],
        'hidden_dim': CONFIG['hidden_dim'],
        'num_layers': CONFIG['num_layers'],
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
}

all_histories = {}

for M in CONFIG['coverage_levels']:
    accuracies, history = run_experiment_for_coverage(
        M, CONFIG, SYMBOL_TO_IDX, IDEAL_VECTORS, device
    )
    
    results['lstm'].append(accuracies['lstm'])
    results['mindist'].append(accuracies['mindist'])
    results['kl'].append(accuracies['kl'])
    results['ml'].append(accuracies['ml'])
    all_histories[M] = history
    
    plot_prefix = f"{CONFIG['error_name']}_eta{CONFIG['eta']}"
    plot_path = os.path.join(CONFIG['results_dir'], f"training_plot_{plot_prefix}_M{M}.png")
    plot_training_history(history, M, plot_path, CONFIG)


# =============================================================================
# CELL 17: SAVE FINAL RESULTS & PLOT
# =============================================================================

print("\n" + "="*70)
print("📊 FINAL RESULTS SUMMARY")
print("="*70)

results_json_path = os.path.join(CONFIG['results_dir'], "experiment_results.json")
with open(results_json_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"💾 Results saved: {results_json_path}")

print(f"\n   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']}")
print(f"   {'M':<8} {'Bi-LSTM':<12} {'Min.Dist':<12} {'KL Div':<12} {'Max.Like':<12}")
print(f"   {'-'*56}")
for i, M in enumerate(results['coverage']):
    print(f"   {M:<8} {results['lstm'][i]:<12.2f} {results['mindist'][i]:<12.2f} "
          f"{results['kl'][i]:<12.2f} {results['ml'][i]:<12.2f}")
print(f"   {'='*56}")

plot_path = os.path.join(CONFIG['results_dir'], "final_comparison_plot.png")
plot_comparison_results(results, plot_path, CONFIG)

print(f"\n✅ All experiments completed!")
print(f"   Eta: {CONFIG['eta']}, Classes: {CONFIG['vocab_size']}")
print(f"   Results directory: {CONFIG['results_dir']}")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080
📋 CONFIGURATION
   Error Model: erlich (EZ17)
   Eta: 0.2, Ell Values: [-2, -1, 0, 1, 2]
   Sequence Length: 136
   Vocab Size: 34 classes
   Theoretical Capacity: 5.0875 bits/position
   Dataset Path: ./dataset/dna_EZ17_eta0.2_100000_50.pkl
   Results Dir: ./results_EZ17_eta0.2
🎲 Random seed set to: 42

📊 Symbol Mappings (η=0.2):
   Total symbols: 34

   Sample mappings (first 10):
   A                0    [1.00, 0.00, 0.00, 0.00]
   C                1    [0.00, 1.00, 0.00, 0.00]
   G                2    [0.00, 0.00, 1.00, 0.00]
   T                3    [0.00, 0.00, 0.00, 1.00]
   B1_ell_neg2      4    [0.10, 0.90, 0.00, 0.00]
   B1_ell_neg1      5    [0.30, 0.70, 0.00, 0.00]
   B1_ell0          6    [0.50, 0.50, 0.00, 0.00]
   B1_ell1          7    [0.70, 0.30, 0.00, 0.00]
   B1_ell2          8    [0.90, 0.10, 0.00, 0.00]
   B2_ell_neg2      9    [0.10, 0.00, 0.90, 0.00]
   ... (24 more)

📦 LOADING DATASET
✅ Dataset loaded: ./datas

/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:149: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


   Epoch 011/100 | Train: 2.7270 | Val: 2.7254 | LR: 1.00e-03 | Time: 37.1s
      EarlyStopping counter: 1/10
   Epoch 012/100 | Train: 2.7258 | Val: 2.7243 | LR: 1.00e-03 | Time: 37.2s
      ✓ Val loss improved (2.7253 → 2.7243). Saving...
   Epoch 013/100 | Train: 2.7252 | Val: 2.7242 | LR: 9.99e-04 | Time: 38.0s
      ✓ Val loss improved (2.7243 → 2.7242). Saving...
   Epoch 014/100 | Train: 2.7249 | Val: 2.7241 | LR: 9.97e-04 | Time: 37.3s
      ✓ Val loss improved (2.7242 → 2.7241). Saving...
   Epoch 015/100 | Train: 2.7242 | Val: 2.7228 | LR: 9.95e-04 | Time: 37.5s
      ✓ Val loss improved (2.7241 → 2.7228). Saving...
   Epoch 016/100 | Train: 2.7239 | Val: 2.7229 | LR: 9.92e-04 | Time: 37.4s
      EarlyStopping counter: 1/10
   Epoch 017/100 | Train: 2.7236 | Val: 2.7223 | LR: 9.89e-04 | Time: 37.3s
      ✓ Val loss improved (2.7228 → 2.7223). Saving...
   Epoch 018/100 | Train: 2.7233 | Val: 2.7223 | LR: 9.85e-04 | Time: 37.2s
      ✓ Val loss improved (2.7223 → 2.7223). Savi

   Epoch 080/100 | Train: 2.7090 | Val: 2.7137 | LR: 1.29e-04 | Time: 43.5s
      EarlyStopping counter: 3/10
   Epoch 081/100 | Train: 2.7089 | Val: 2.7126 | LR: 1.18e-04 | Time: 37.5s
      EarlyStopping counter: 4/10
   Epoch 082/100 | Train: 2.7087 | Val: 2.7127 | LR: 1.07e-04 | Time: 38.1s
      EarlyStopping counter: 5/10
   Epoch 083/100 | Train: 2.7088 | Val: 2.7126 | LR: 9.64e-05 | Time: 37.4s
      EarlyStopping counter: 6/10
   Epoch 084/100 | Train: 2.7085 | Val: 2.7128 | LR: 8.64e-05 | Time: 44.8s
      EarlyStopping counter: 7/10
   Epoch 085/100 | Train: 2.7084 | Val: 2.7134 | LR: 7.69e-05 | Time: 37.6s
      EarlyStopping counter: 8/10
   Epoch 086/100 | Train: 2.7083 | Val: 2.7129 | LR: 6.79e-05 | Time: 37.4s
      EarlyStopping counter: 9/10
   Epoch 087/100 | Train: 2.7082 | Val: 2.7131 | LR: 5.95e-05 | Time: 37.6s
      EarlyStopping counter: 10/10

   🛑 Early stopping triggered at epoch 87
   💾 Final model saved: ./results_EZ17_eta0.2/final_model_EZ17_eta0.2_M1.pth

   Epoch 053/100 | Train: 2.2022 | Val: 2.2043 | LR: 5.53e-04 | Time: 66.5s
      EarlyStopping counter: 1/10
   Epoch 054/100 | Train: 2.2022 | Val: 2.2046 | LR: 5.35e-04 | Time: 66.0s
      EarlyStopping counter: 2/10
   Epoch 055/100 | Train: 2.2017 | Val: 2.2030 | LR: 5.18e-04 | Time: 66.0s
      ✓ Val loss improved (2.2037 → 2.2030). Saving...
   Epoch 056/100 | Train: 2.2010 | Val: 2.2028 | LR: 5.00e-04 | Time: 67.1s
      ✓ Val loss improved (2.2030 → 2.2028). Saving...
   Epoch 057/100 | Train: 2.2012 | Val: 2.2043 | LR: 4.83e-04 | Time: 65.9s
      EarlyStopping counter: 1/10
   Epoch 058/100 | Train: 2.2007 | Val: 2.2025 | LR: 4.66e-04 | Time: 66.5s
      ✓ Val loss improved (2.2028 → 2.2025). Saving...
   Epoch 059/100 | Train: 2.2003 | Val: 2.2025 | LR: 4.48e-04 | Time: 65.4s
      ✓ Val loss improved (2.2025 → 2.2025). Saving...
   Epoch 060/100 | Train: 2.2004 | Val: 2.2024 | LR: 4.31e-04 | Time: 65.9s
      ✓ Val loss improved (2.2025 → 2.2024). Saving...
   Epoch 061/10

...

   Epoch 054/100 | Train: 0.5030 | Val: 0.4974 | LR: 5.35e-04 | Time: 898.7s
      EarlyStopping counter: 1/10
   Epoch 055/100 | Train: 0.5025 | Val: 0.4971 | LR: 5.18e-04 | Time: 828.5s
      ✓ Val loss improved (0.4973 → 0.4971). Saving...
   Epoch 056/100 | Train: 0.5022 | Val: 0.4968 | LR: 5.00e-04 | Time: 820.4s
      ✓ Val loss improved (0.4971 → 0.4968). Saving...
   Epoch 057/100 | Train: 0.5020 | Val: 0.4967 | LR: 4.83e-04 | Time: 818.1s
      ✓ Val loss improved (0.4968 → 0.4967). Saving...
   Epoch 058/100 | Train: 0.5016 | Val: 0.4965 | LR: 4.66e-04 | Time: 853.3s
      ✓ Val loss improved (0.4967 → 0.4965). Saving...
   Epoch 059/100 | Train: 0.5014 | Val: 0.4961 | LR: 4.48e-04 | Time: 808.9s
      ✓ Val loss improved (0.4965 → 0.4961). Saving...
   Epoch 060/100 | Train: 0.5011 | Val: 0.4961 | LR: 4.31e-04 | Time: 809.5s
      ✓ Val loss improved (0.4961 → 0.4961). Saving...
   Epoch 061/100 | Train: 0.5009 | Val: 0.4960 | LR: 4.14e-04 | Time: 811.2s
      ✓ Val loss imp

   Epoch 013/100 | Train: 0.4359 | Val: 0.3996 | LR: 9.99e-04 | Time: 1101.9s
      ✓ Val loss improved (0.4106 → 0.3996). Saving...
   Epoch 014/100 | Train: 0.4252 | Val: 0.3937 | LR: 9.97e-04 | Time: 1088.4s
      ✓ Val loss improved (0.3996 → 0.3937). Saving...
   Epoch 015/100 | Train: 0.4172 | Val: 0.3882 | LR: 9.95e-04 | Time: 1088.3s
      ✓ Val loss improved (0.3937 → 0.3882). Saving...
   Epoch 016/100 | Train: 0.4108 | Val: 0.3865 | LR: 9.92e-04 | Time: 1086.0s
      ✓ Val loss improved (0.3882 → 0.3865). Saving...
   Epoch 017/100 | Train: 0.4058 | Val: 0.3846 | LR: 9.89e-04 | Time: 1083.8s
      ✓ Val loss improved (0.3865 → 0.3846). Saving...
   Epoch 018/100 | Train: 0.4014 | Val: 0.3797 | LR: 9.85e-04 | Time: 1086.7s
      ✓ Val loss improved (0.3846 → 0.3797). Saving...
   Epoch 019/100 | Train: 0.3986 | Val: 0.3775 | LR: 9.81e-04 | Time: 1083.9s
      ✓ Val loss improved (0.3797 → 0.3775). Saving...
   Epoch 020/100 | Train: 0.3954 | Val: 0.3768 | LR: 9.76e-04 | Time:

   Epoch 078/100 | Train: 0.3596 | Val: 0.3544 | LR: 1.54e-04 | Time: 1080.5s
      EarlyStopping counter: 1/10
   Epoch 079/100 | Train: 0.3594 | Val: 0.3539 | LR: 1.41e-04 | Time: 1083.3s
      ✓ Val loss improved (0.3541 → 0.3539). Saving...
   Epoch 080/100 | Train: 0.3593 | Val: 0.3540 | LR: 1.29e-04 | Time: 1083.7s
      EarlyStopping counter: 1/10
   Epoch 081/100 | Train: 0.3592 | Val: 0.3539 | LR: 1.18e-04 | Time: 1085.6s
      EarlyStopping counter: 2/10
   Epoch 082/100 | Train: 0.3591 | Val: 0.3540 | LR: 1.07e-04 | Time: 1091.5s
      EarlyStopping counter: 3/10
   Epoch 083/100 | Train: 0.3589 | Val: 0.3538 | LR: 9.64e-05 | Time: 1083.1s
      ✓ Val loss improved (0.3539 → 0.3538). Saving...
   Epoch 084/100 | Train: 0.3588 | Val: 0.3537 | LR: 8.64e-05 | Time: 1081.7s
      ✓ Val loss improved (0.3538 → 0.3537). Saving...
   Epoch 085/100 | Train: 0.3588 | Val: 0.3536 | LR: 7.69e-05 | Time: 1115.8s
      ✓ Val loss improved (0.3537 → 0.3536). Saving...
   Epoch 086/100 | T

   Epoch 037/100 | Train: 0.2770 | Val: 0.2678 | LR: 8.08e-04 | Time: 1397.1s
      EarlyStopping counter: 1/10
   Epoch 038/100 | Train: 0.2766 | Val: 0.2681 | LR: 7.94e-04 | Time: 1378.6s
      EarlyStopping counter: 2/10
   Epoch 039/100 | Train: 0.2757 | Val: 0.2661 | LR: 7.80e-04 | Time: 1363.4s
      ✓ Val loss improved (0.2676 → 0.2661). Saving...
   Epoch 040/100 | Train: 0.2751 | Val: 0.2656 | LR: 7.65e-04 | Time: 1368.9s
      ✓ Val loss improved (0.2661 → 0.2656). Saving...
   Epoch 041/100 | Train: 0.2747 | Val: 0.2654 | LR: 7.50e-04 | Time: 1400.0s
      ✓ Val loss improved (0.2656 → 0.2654). Saving...
   Epoch 042/100 | Train: 0.2741 | Val: 0.2657 | LR: 7.35e-04 | Time: 1407.6s
      EarlyStopping counter: 1/10
   Epoch 043/100 | Train: 0.2736 | Val: 0.2648 | LR: 7.19e-04 | Time: 1366.8s
      ✓ Val loss improved (0.2654 → 0.2648). Saving...
   Epoch 044/100 | Train: 0.2731 | Val: 0.2642 | LR: 7.04e-04 | Time: 1364.4s
      ✓ Val loss improved (0.2648 → 0.2642). Saving...


   ✅ RESULTS M=50 (η=0.2):
      Bi-LSTM:         90.10%
      Min. Distance:   83.71%
      KL Divergence:   83.13%
      Max. Likelihood: 83.13%
   📈 Training plot saved: ./results_EZ17_eta0.2/training_plot_EZ17_eta0.2_M50.png

📊 FINAL RESULTS SUMMARY
💾 Results saved: ./results_EZ17_eta0.2/experiment_results.json

   Eta: 0.2, Vocab Size: 34
   M        Bi-LSTM      Min.Dist     KL Div       Max.Like    
   --------------------------------------------------------
   1        11.23        11.30        11.30        11.30       
   2        19.59        19.25        19.25        19.25       
   3        26.17        25.06        25.06        25.06       
   5        34.16        31.60        31.91        31.91       
   8        47.50        44.32        44.33        44.33       
   10       52.51        47.88        48.26        48.26       
   15       63.71        57.56        57.89        57.89       
   20       71.27        62.05        64.26        64.26       
   25       76.67